# Solution B1 · California housing: baseline, linear model and logarithmic target

**Type**: Implement with scikit-learn

## Objective

Build a linear model for house prices in California, evaluate it against a baseline, and find out that
transforming the target into a logarithm changes which model comes out best **depending on the metric**.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import train_test_split

X, y = ds.california()
print(X.shape, "| min/max price:", y.min(), y.max())

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
rmse_base = mean_squared_error(y_te, base.predict(X_te), squared=False)
print("baseline RMSE:", round(rmse_base, 3))

In [ ]:
linear = LinearRegression().fit(X_tr, y_tr)
pred_lin = linear.predict(X_te)
rmse_lin = mean_squared_error(y_te, pred_lin, squared=False)
mae_lin = mean_absolute_error(y_te, pred_lin)
mape_lin = mean_absolute_percentage_error(y_te, pred_lin)
r2_lin = r2_score(y_te, pred_lin)

`TransformedTargetRegressor` applies `func` to the target before the fit and `inverse_func` to the predictions: the metrics
therefore stay in the original unit (dollars) and are comparable with those of the linear model. Comparing the R²
computed on the logarithm with the one computed on the prices would be a mistake: they are two different targets.

In [ ]:
log_linear = TransformedTargetRegressor(regressor=LinearRegression(), func=np.log, inverse_func=np.exp)
log_linear.fit(X_tr, y_tr)
pred_log = log_linear.predict(X_te)
rmse_log = mean_squared_error(y_te, pred_log, squared=False)
mae_log = mean_absolute_error(y_te, pred_log)
mape_log = mean_absolute_percentage_error(y_te, pred_log)

print(f"linear : RMSE {rmse_lin:.3f} | MAE {mae_lin:.3f} | MAPE {mape_lin:.3f}")
print(f"log    : RMSE {rmse_log:.3f} | MAE {mae_log:.3f} | MAPE {mape_log:.3f}")
best_by_rmse = "linear" if rmse_lin < rmse_log else "log"
best_by_mape = "linear" if mape_lin < mape_log else "log"
print("best by RMSE:", best_by_rmse, "| best by MAPE:", best_by_mape)

In [ ]:
capped = (y_te >= 5.0)
capped_share = capped.mean()
rmse_capped = mean_squared_error(y_te[capped], pred_lin[capped], squared=False)
rmse_others = mean_squared_error(y_te[~capped], pred_lin[~capped], squared=False)
print(f"capped houses: {capped_share:.1%} | RMSE on those {rmse_capped:.2f} versus {rmse_others:.2f} on the others")

## Check

In [ ]:
with verify():
    check(len(X_te) == round(0.2 * len(X)), "the split is 80/20", "X_te should have 20% of the rows")
    check(rmse_lin < rmse_base, "the linear model beats the baseline",
          "the linear model should have a lower RMSE than the baseline")
    check(close(rmse_lin, mean_squared_error(y_te, linear.predict(X_te), squared=False)),
          "rmse_lin is computed correctly", "rmse_lin does not match the predictions of `linear`")
    check(close(rmse_log, mean_squared_error(y_te, log_linear.predict(X_te), squared=False)),
          "rmse_log is computed in the original units",
          "rmse_log does not match log_linear.predict(X_te): the predictions must already be in the original unit")
    check(log_linear.predict(X_te).min() > 0, "TransformedTargetRegressor returns positive prices",
          "with func=np.log and inverse_func=np.exp the predictions must be positive")
    check(best_by_rmse == ("linear" if rmse_lin < rmse_log else "log"), "choice consistent with the RMSE",
          "best_by_rmse is not consistent with rmse_lin and rmse_log")
    check(best_by_mape == ("linear" if mape_lin < mape_log else "log"), "choice consistent with the MAPE",
          "best_by_mape is not consistent with mape_lin and mape_log")
    check(abs(capped_share - (y_te >= 5.0).mean()) < 1e-9, "correct share of capped houses",
          "capped_share must be the mean of the mask (y_te >= 5.0)")
    if not ds.CALIFORNIA_SYNTHETIC:
        check(best_by_rmse != best_by_mape,
              "the two criteria choose DIFFERENT models: that is the point of the exercise",
              "with RMSE and MAPE you should find two different winners: review the calculations, all in the original unit")
        check(rmse_capped > 1.5 * rmse_others, "on the capped houses the model is much more wrong",
              "the RMSE on the capped houses should be much higher")
done()

## What you learned

- The model on the logarithm has a **better MAPE** (it is less wrong in percentage terms) but a **worse RMSE**. There is no
  contradiction: the squared RMSE weighs mostly the errors on expensive houses, where a small
  percentage error is worth many dollars; the MAPE treats a 100 thousand house and a 500 thousand one the same way.
- `exp(mean of the logarithms)` is about the **median**, not the mean: bringing the predictions back to the original unit
  introduces a systematic bias (retransformation bias), whose sign and size depend on the sample.
- The capping of the target at 5.0 is not visible in any aggregate metric, but the model is enormously
  wrong on those houses: it is a limit of the data, not of the model.

## Common pitfall

Comparing two models with metrics computed in different units (one R² on the log, another on the prices) or choosing the
target transformation by looking at the metric that the transformation itself optimizes. The right question is always:
which error costs more in the real problem?

## Interview question

<details><summary>The model on the logarithm of the price has a better MAPE but a worse RMSE. Which one do you put in production?</summary>

It depends on what the error costs the company. If the relative error matters (for example an estimate shown to the customer,
where being off by 30% is bad on both a small and a large house) I choose the model on the logarithm. If
the absolute error in euros matters (for example the committed capital) and expensive houses weigh more, I look at the RMSE or
the error weighted by value. In any case I would check the bias of the predictions brought back to the original unit.

</details>